# OpenResty Slowdown Attribution

This notebook is the main attribution notebook for the lab. It is built to answer one question:

**When latency rises, is the slowdown coming from the client view, the edge/OpenResty layer, or the origin?**

The workflow is:

1. Check scrape health and whether the time window contains data.
2. Start from the client-side `k6` latency and rate metrics.
3. Compare them with edge request latency and edge upstream latency.
4. Break the OpenResty path into access/content/header-filter/log phases plus Redis and metadata calls.
5. Compare baseline live/VOD client latency with origin latency.
6. Validate the attribution using raw edge access logs.

The notebook uses the Prometheus Python wrapper in [prometheus_tools.py](/workspaces/Development/K6_Playground/python/src/prometheus_tools.py), so the same query approach also works in scripts.

In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from pathlib import Path
import os
import sys

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "python" / "src").exists():
    for candidate in ROOT.parents:
        if (candidate / "python" / "src").exists():
            ROOT = candidate
            break

sys.path.insert(0, str(ROOT / "python" / "src"))

from prometheus_tools import instant_query, query_range

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)


In [ ]:
LOOKBACK_MINUTES = int(os.getenv("LOOKBACK_MINUTES", "30"))
STEP = os.getenv("PROM_STEP", "15s")
PROMETHEUS_URL = os.getenv("PROMETHEUS_URL", "http://localhost:9090")
LOG_PATH = ROOT / "edge" / "logs" / "edge-access.log"

end = datetime.now(timezone.utc)
start = end - timedelta(minutes=LOOKBACK_MINUTES)

BASELINE_LAB = "edge_cache_baseline"
OPENRESTY_LAB = "openresty_runtime"

EDGE_TARGET_MAP = {
    "live_master": ("live", "live_master"),
    "live_playlist": ("live", "live_playlist"),
    "live_segment": ("live", "live_segment"),
    "vod_manifest": ("vod", "vod_manifest"),
    "vod_playlist": ("vod", "vod_playlist"),
    "vod_segment": ("vod", "vod_segment"),
    "openresty_transform": ("openresty", "lua_transform"),
    "openresty_policy": ("openresty", "lua_policy"),
    "openresty_gc_probe": ("openresty", "lua_transform"),
}

ORIGIN_PATTERNS = {
    "live_master": r"/live/.*/master\\.m3u8",
    "live_playlist": r"/live/.*/live\\.m3u8",
    "live_segment": r"/live/.*/seg_.*\\.ts",
    "vod_manifest": r"/vod/.*/master\\.m3u8",
    "vod_playlist": r"/vod/.*/playlist\\.m3u8",
    "vod_segment": r"/vod/.*/seg_.*\\.ts",
}

def q(expression: str, step: str = STEP) -> pd.DataFrame:
    try:
        return query_range(expression, start, end, step=step, prometheus_url=PROMETHEUS_URL)
    except RuntimeError as exc:
        print(f"Query failed: {expression}\n{exc}")
        return pd.DataFrame(columns=["timestamp", "value"])

def qi(expression: str) -> pd.DataFrame:
    try:
        return instant_query(expression, prometheus_url=PROMETHEUS_URL)
    except RuntimeError as exc:
        print(f"Instant query failed: {expression}\n{exc}")
        return pd.DataFrame(columns=["timestamp", "value"])

def with_ms(frame: pd.DataFrame, column: str = "value") -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    enriched = frame.copy()
    enriched[f"{column}_ms"] = enriched[column] * 1000.0
    return enriched

def latest_by(frame: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    return frame.sort_values("timestamp").groupby(group_cols, dropna=False).tail(1).reset_index(drop=True)

def latest_subset(frame: pd.DataFrame, group_cols: list[str], columns: list[str]) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=columns)
    latest = latest_by(frame, group_cols)
    for column in columns:
        if column not in latest.columns:
            latest[column] = np.nan
    return latest[columns].copy()

def add_constant_label(frame: pd.DataFrame, name: str, value: str) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    enriched = frame.copy()
    enriched[name] = value
    return enriched

def normalize_upstream_value(series: pd.Series) -> pd.Series:
    text = series.fillna("").astype(str).replace({"": np.nan, "-": np.nan})
    first_value = text.str.split(",").str[0]
    return pd.to_numeric(first_value, errors="coerce")

def classify_summary_row(row: pd.Series) -> str:
    client_p95 = row.get("client_p95_ms")
    edge_p95 = row.get("edge_p95_ms")
    upstream_p95 = row.get("upstream_p95_ms")
    origin_p95 = row.get("origin_p95_ms")
    route_family = row.get("route_family")

    if pd.isna(client_p95):
        return "no_recent_client_signal"

    if route_family == "openresty":
        if pd.notna(edge_p95) and edge_p95 >= 0.75 * client_p95:
            return "edge_openresty_path"
        return "client_or_measurement_gap"

    if (
        pd.notna(upstream_p95)
        and pd.notna(edge_p95)
        and pd.notna(client_p95)
        and edge_p95 >= 0.70 * client_p95
        and upstream_p95 >= 0.70 * edge_p95
    ):
        if pd.notna(origin_p95) and origin_p95 >= 0.70 * upstream_p95:
            return "origin"
        return "upstream_or_origin_adjacent"

    if pd.notna(edge_p95) and edge_p95 >= 0.75 * client_p95:
        return "edge"

    if pd.notna(edge_p95) and client_p95 >= 1.25 * edge_p95:
        return "client_network_or_load_driver"

    return "mixed_or_low_signal"

start, end, STEP, PROMETHEUS_URL, LOG_PATH


In [ ]:
up_df = qi("up")

client_rate_df = q(
    f'sum by (lab, name) (rate(k6_http_reqs_total{{lab=~"{BASELINE_LAB}|{OPENRESTY_LAB}"}}[1m]))'
)
client_p95_df = with_ms(
    q(f'k6_http_req_duration_p95{{lab=~"{BASELINE_LAB}|{OPENRESTY_LAB}"}}')
)

edge_request_p95_df = with_ms(
    q('histogram_quantile(0.95, sum by (le, route_family, target_kind) (rate(openresty_edge_request_duration_seconds_bucket[5m])))')
)
edge_request_avg_df = with_ms(
    q('sum by (route_family, target_kind) (rate(openresty_edge_request_duration_seconds_sum[1m])) / clamp_min(sum by (route_family, target_kind) (rate(openresty_edge_request_duration_seconds_count[1m])), 1e-9)')
)
edge_upstream_p95_df = with_ms(
    q('histogram_quantile(0.95, sum by (le, route_family, target_kind) (rate(openresty_edge_upstream_response_seconds_bucket[5m])))')
)
edge_upstream_avg_df = with_ms(
    q('sum by (route_family, target_kind) (rate(openresty_edge_upstream_response_seconds_sum[1m])) / clamp_min(sum by (route_family, target_kind) (rate(openresty_edge_upstream_response_seconds_count[1m])), 1e-9)')
)

edge_access_p95_df = with_ms(
    q('histogram_quantile(0.95, sum by (le, target_kind) (rate(openresty_edge_lua_access_duration_seconds_bucket{route_family="openresty"}[5m])))')
)
edge_content_p95_df = with_ms(
    q('histogram_quantile(0.95, sum by (le, target_kind) (rate(openresty_edge_lua_content_duration_seconds_bucket{route_family="openresty"}[5m])))')
)
edge_header_p95_df = with_ms(
    q('histogram_quantile(0.95, sum by (le, target_kind) (rate(openresty_edge_header_filter_duration_seconds_bucket{route_family="openresty"}[5m])))')
)
edge_log_p95_df = with_ms(
    q('histogram_quantile(0.95, sum by (le, target_kind) (rate(openresty_edge_lua_log_duration_seconds_bucket{route_family="openresty"}[5m])))')
)

metadata_avg_df = with_ms(
    q('sum by (target_kind, outcome) (rate(openresty_edge_metadata_duration_seconds_sum{route_family="openresty"}[1m])) / clamp_min(sum by (target_kind, outcome) (rate(openresty_edge_metadata_duration_seconds_count{route_family="openresty"}[1m])), 1e-9)')
)
redis_connect_avg_df = with_ms(
    q('sum by (target_kind, phase, outcome) (rate(openresty_edge_redis_connect_duration_seconds_sum{route_family="openresty"}[1m])) / clamp_min(sum by (target_kind, phase, outcome) (rate(openresty_edge_redis_connect_duration_seconds_count{route_family="openresty"}[1m])), 1e-9)')
)
redis_command_avg_df = with_ms(
    q('sum by (target_kind, phase, operation, outcome) (rate(openresty_edge_redis_command_duration_seconds_sum{route_family="openresty"}[1m])) / clamp_min(sum by (target_kind, phase, operation, outcome) (rate(openresty_edge_redis_command_duration_seconds_count{route_family="openresty"}[1m])), 1e-9)')
)
kafka_publish_avg_df = with_ms(
    q('sum by (target_kind, outcome) (rate(openresty_edge_kafka_mock_publish_duration_seconds_sum{route_family="openresty"}[1m])) / clamp_min(sum by (target_kind, outcome) (rate(openresty_edge_kafka_mock_publish_duration_seconds_count{route_family="openresty"}[1m])), 1e-9)')
)

{
    "scrape_targets": 0 if up_df.empty else len(up_df),
    "client_series": len(client_p95_df),
    "edge_request_series": len(edge_request_p95_df),
    "origin_placeholder": "loaded next cell",
}


In [ ]:
origin_p95_frames = []
origin_avg_frames = []
origin_rate_frames = []

for origin_kind, pattern in ORIGIN_PATTERNS.items():
    p95_frame = with_ms(
        q(f'histogram_quantile(0.95, sum by (le) (rate(origin_request_latency_seconds_bucket{{path=~"{pattern}"}}[5m])))')
    )
    if not p95_frame.empty:
        origin_p95_frames.append(add_constant_label(p95_frame, "origin_kind", origin_kind))

    avg_frame = with_ms(
        q(f'sum(rate(origin_request_latency_seconds_sum{{path=~"{pattern}"}}[1m])) / clamp_min(sum(rate(origin_request_latency_seconds_count{{path=~"{pattern}"}}[1m])), 1e-9)')
    )
    if not avg_frame.empty:
        origin_avg_frames.append(add_constant_label(avg_frame, "origin_kind", origin_kind))

    rate_frame = q(f'sum(rate(origin_requests_total{{path=~"{pattern}"}}[1m]))')
    if not rate_frame.empty:
        origin_rate_frames.append(add_constant_label(rate_frame, "origin_kind", origin_kind))

origin_p95_df = pd.concat(origin_p95_frames, ignore_index=True) if origin_p95_frames else pd.DataFrame(columns=["timestamp", "value", "value_ms", "origin_kind"])
origin_avg_df = pd.concat(origin_avg_frames, ignore_index=True) if origin_avg_frames else pd.DataFrame(columns=["timestamp", "value", "value_ms", "origin_kind"])
origin_rate_df = pd.concat(origin_rate_frames, ignore_index=True) if origin_rate_frames else pd.DataFrame(columns=["timestamp", "value", "origin_kind"])

health_df = up_df.copy()
if not health_df.empty:
    health_df["status"] = health_df["value"].map({1.0: "up", 0.0: "down"}).fillna("unknown")

display(health_df)


In [ ]:
client_latest = latest_subset(client_p95_df, ["lab", "name"], ["lab", "name", "value_ms"]).rename(columns={"value_ms": "client_p95_ms"})
client_latest["route_family"] = client_latest["name"].map(lambda value: EDGE_TARGET_MAP.get(value, (None, None))[0])
client_latest["target_kind"] = client_latest["name"].map(lambda value: EDGE_TARGET_MAP.get(value, (None, None))[1])

edge_p95_latest = latest_subset(edge_request_p95_df, ["route_family", "target_kind"], ["route_family", "target_kind", "value_ms"]).rename(columns={"value_ms": "edge_p95_ms"})
edge_avg_latest = latest_subset(edge_request_avg_df, ["route_family", "target_kind"], ["route_family", "target_kind", "value_ms"]).rename(columns={"value_ms": "edge_avg_ms"})
upstream_p95_latest = latest_subset(edge_upstream_p95_df, ["route_family", "target_kind"], ["route_family", "target_kind", "value_ms"]).rename(columns={"value_ms": "upstream_p95_ms"})
upstream_avg_latest = latest_subset(edge_upstream_avg_df, ["route_family", "target_kind"], ["route_family", "target_kind", "value_ms"]).rename(columns={"value_ms": "upstream_avg_ms"})
origin_p95_latest = latest_subset(origin_p95_df, ["origin_kind"], ["origin_kind", "value_ms"]).rename(columns={"origin_kind": "name", "value_ms": "origin_p95_ms"})
origin_avg_latest = latest_subset(origin_avg_df, ["origin_kind"], ["origin_kind", "value_ms"]).rename(columns={"origin_kind": "name", "value_ms": "origin_avg_ms"})
client_rate_latest = latest_subset(client_rate_df, ["lab", "name"], ["lab", "name", "value"]).rename(columns={"value": "client_rate_rps"})
origin_rate_latest = latest_subset(origin_rate_df, ["origin_kind"], ["origin_kind", "value"]).rename(columns={"origin_kind": "name", "value": "origin_rate_rps"})

summary_df = (
    client_latest
    .merge(client_rate_latest, on=["lab", "name"], how="left")
    .merge(edge_p95_latest, on=["route_family", "target_kind"], how="left")
    .merge(edge_avg_latest, on=["route_family", "target_kind"], how="left")
    .merge(upstream_p95_latest, on=["route_family", "target_kind"], how="left")
    .merge(upstream_avg_latest, on=["route_family", "target_kind"], how="left")
    .merge(origin_p95_latest, on="name", how="left")
    .merge(origin_avg_latest, on="name", how="left")
    .merge(origin_rate_latest, on="name", how="left")
)

summary_df["likely_bottleneck"] = summary_df.apply(classify_summary_row, axis=1)
summary_df = summary_df.sort_values(["lab", "client_p95_ms"], ascending=[True, False]).reset_index(drop=True)

display(summary_df)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))

rate_plot_df = client_rate_df.copy()
if not rate_plot_df.empty:
    sns.lineplot(data=rate_plot_df, x="timestamp", y="value", hue="name", style="lab", ax=axes[0, 0])
axes[0, 0].set_title("Client Request Rate By k6 Name")
axes[0, 0].set_ylabel("requests/sec")

latency_plot_df = summary_df.melt(
    id_vars=["lab", "name"],
    value_vars=["client_p95_ms", "edge_p95_ms", "upstream_p95_ms", "origin_p95_ms"],
    var_name="layer",
    value_name="ms",
).dropna()
sns.barplot(data=latency_plot_df, x="name", y="ms", hue="layer", ax=axes[0, 1])
axes[0, 1].set_title("Latest P95 By Layer")
axes[0, 1].tick_params(axis="x", rotation=25)

origin_plot_df = origin_p95_df.copy()
if not origin_plot_df.empty:
    sns.lineplot(data=origin_plot_df, x="timestamp", y="value_ms", hue="origin_kind", ax=axes[1, 0])
axes[1, 0].set_title("Origin P95 Latency By Path Family")
axes[1, 0].set_ylabel("ms")

bottleneck_counts = summary_df["likely_bottleneck"].value_counts().rename_axis("likely_bottleneck").reset_index(name="count")
sns.barplot(data=bottleneck_counts, x="likely_bottleneck", y="count", ax=axes[1, 1])
axes[1, 1].set_title("Latest Attribution Classification")
axes[1, 1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
phase_latest_frames = []
for frame, metric_name in [
    (edge_access_p95_df, "access_p95_ms"),
    (edge_content_p95_df, "content_p95_ms"),
    (edge_header_p95_df, "header_filter_p95_ms"),
    (edge_log_p95_df, "log_p95_ms"),
]:
    if not frame.empty:
        latest = latest_by(frame, ["target_kind"])[["target_kind", "value_ms"]].rename(columns={"value_ms": metric_name})
        phase_latest_frames.append(latest.set_index("target_kind"))

phase_summary_df = pd.concat(phase_latest_frames, axis=1).reset_index() if phase_latest_frames else pd.DataFrame()
metadata_latest_df = latest_by(metadata_avg_df, ["target_kind", "outcome"])[["target_kind", "outcome", "value_ms"]].rename(columns={"value_ms": "metadata_avg_ms"})
redis_connect_latest_df = latest_by(redis_connect_avg_df, ["target_kind", "phase", "outcome"])[["target_kind", "phase", "outcome", "value_ms"]].rename(columns={"value_ms": "redis_connect_avg_ms"})
redis_command_latest_df = latest_by(redis_command_avg_df, ["target_kind", "phase", "operation", "outcome"])[["target_kind", "phase", "operation", "outcome", "value_ms"]].rename(columns={"value_ms": "redis_command_avg_ms"})
kafka_latest_df = latest_by(kafka_publish_avg_df, ["target_kind", "outcome"])[["target_kind", "outcome", "value_ms"]].rename(columns={"value_ms": "kafka_publish_avg_ms"})

display(phase_summary_df)
display(metadata_latest_df)
display(redis_connect_latest_df)
display(redis_command_latest_df)
display(kafka_latest_df)

fig, axes = plt.subplots(2, 2, figsize=(20, 12))

if not phase_summary_df.empty:
    phase_plot_df = phase_summary_df.melt(id_vars=["target_kind"], var_name="phase_metric", value_name="ms").dropna()
    sns.barplot(data=phase_plot_df, x="target_kind", y="ms", hue="phase_metric", ax=axes[0, 0])
axes[0, 0].set_title("OpenResty Phase P95")

if not metadata_latest_df.empty:
    sns.barplot(data=metadata_latest_df, x="target_kind", y="metadata_avg_ms", hue="outcome", ax=axes[0, 1])
axes[0, 1].set_title("Metadata Average")

if not redis_connect_latest_df.empty:
    sns.barplot(data=redis_connect_latest_df, x="phase", y="redis_connect_avg_ms", hue="target_kind", ax=axes[1, 0])
axes[1, 0].set_title("Redis Connect Average")

if not redis_command_latest_df.empty:
    redis_command_plot_df = redis_command_latest_df[redis_command_latest_df["operation"].isin(["ping", "incr", "hmset", "hgetall"])]
    sns.barplot(data=redis_command_plot_df, x="operation", y="redis_command_avg_ms", hue="phase", ax=axes[1, 1])
axes[1, 1].set_title("Redis Command Average")

for axis in axes.flat:
    axis.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
edge_log_df = pd.read_json(LOG_PATH, lines=True)
edge_log_df["time"] = pd.to_datetime(edge_log_df["time"], utc=True)
recent_log_df = edge_log_df.loc[edge_log_df["time"] >= start].copy()

numeric_columns = [
    "request_time",
    "lua_access_ms",
    "lua_content_ms",
    "header_filter_ms",
    "metadata_fetch_ms",
    "redis_access_connect_ms",
    "redis_access_command_ms",
    "redis_content_connect_ms",
    "redis_content_command_ms",
    "lua_gc_delta_kb",
]
for column in numeric_columns:
    if column in recent_log_df.columns:
        recent_log_df[column] = pd.to_numeric(recent_log_df[column], errors="coerce")

recent_log_df["request_ms"] = recent_log_df["request_time"] * 1000.0
recent_log_df["upstream_ms"] = normalize_upstream_value(recent_log_df.get("upstream_response_time", pd.Series(dtype=float))) * 1000.0
recent_log_df["edge_overhead_ms"] = (recent_log_df["request_ms"] - recent_log_df["upstream_ms"].fillna(0)).clip(lower=0)
recent_log_df["redis_total_ms"] = recent_log_df[["redis_access_connect_ms", "redis_access_command_ms", "redis_content_connect_ms", "redis_content_command_ms"]].fillna(0).sum(axis=1)
recent_log_df["openresty_internal_ms"] = recent_log_df[["lua_access_ms", "lua_content_ms", "header_filter_ms"]].fillna(0).sum(axis=1)

def classify_log_row(row: pd.Series) -> str:
    if row.get("route_family") == "openresty":
        components = {
            "metadata": row.get("metadata_fetch_ms", 0) or 0,
            "redis": row.get("redis_total_ms", 0) or 0,
            "lua": row.get("openresty_internal_ms", 0) or 0,
        }
        dominant = max(components, key=components.get)
        return f"openresty_{dominant}"

    request_ms = row.get("request_ms", np.nan)
    upstream_ms = row.get("upstream_ms", np.nan)
    edge_overhead_ms = row.get("edge_overhead_ms", np.nan)

    if pd.notna(upstream_ms) and pd.notna(request_ms) and upstream_ms >= max(5.0, 0.70 * request_ms):
        return "origin_or_upstream"
    if pd.notna(edge_overhead_ms) and pd.notna(request_ms) and edge_overhead_ms >= max(5.0, 0.40 * request_ms):
        return "edge_overhead"
    if row.get("upstream_cache_status") == "HIT":
        return "cache_hit_fast_path"
    return "mixed_or_fast"

recent_log_df["likely_request_bottleneck"] = recent_log_df.apply(classify_log_row, axis=1)

slowest_requests_df = recent_log_df.sort_values("request_ms", ascending=False).head(20)
bottleneck_log_summary_df = recent_log_df.groupby(["route_family", "target_kind", "likely_request_bottleneck"], dropna=False).agg(
    samples=("request_ms", "size"),
    avg_request_ms=("request_ms", "mean"),
    p95_request_ms=("request_ms", lambda values: values.quantile(0.95)),
).reset_index().sort_values("p95_request_ms", ascending=False)

display(slowest_requests_df[[
    "time", "uri", "route_family", "target_kind", "status", "request_ms", "upstream_ms",
    "lua_access_ms", "lua_content_ms", "header_filter_ms", "metadata_fetch_ms", "redis_total_ms",
    "upstream_cache_status", "likely_request_bottleneck"
]])
display(bottleneck_log_summary_df.head(20))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.boxplot(data=recent_log_df, x="route_family", y="request_ms", ax=axes[0])
axes[0].set_title("Edge Log Request Time By Route Family")
axes[0].set_ylabel("ms")

top_bottlenecks = bottleneck_log_summary_df.head(12)
sns.barplot(data=top_bottlenecks, x="target_kind", y="p95_request_ms", hue="likely_request_bottleneck", ax=axes[1])
axes[1].set_title("Top Logged Bottlenecks")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


## Troubleshooting Procedure

Use this same order every time so you do not jump straight into Redis or Lua without first checking where the slowdown is visible.

1. **Check target health first**
   - Query `up`.
   - If `origin`, `edge`, or `prometheus` is down, stop there and fix collection first.

2. **Find the client symptom**
   - Query `k6_http_req_duration_p95` filtered by `lab` and `name`.
   - If the client metric is not high, there is no user-visible slowdown yet.

3. **Compare client to edge**
   - Compare `client_p95_ms` to `edge_p95_ms`.
   - If both are high, the slowdown is inside or behind the edge.
   - If only the client is high, suspect client-side timing, k6 pressure, or something outside the edge/origin path.

4. **Split edge from upstream/origin**
   - Compare `edge_p95_ms` to `upstream_p95_ms`.
   - If upstream is most of the edge time, inspect origin.
   - If edge time is high but upstream is low, the problem is at the edge.
   - Important: upstream metrics only exist for requests that actually touched origin, so cache-heavy HIT populations can make upstream p95 look higher than the overall edge p95. Use the log tables to confirm cache status before blaming origin.

5. **If the route family is `openresty`, drill into phases**
   - Check access/content/header-filter/log phase latency.
   - Then check metadata, Redis connect, Redis commands, and Kafka publish.
   - Redis spikes in `access` or `content` point to synchronous request-path delays.
   - Redis spikes in `header_filter_timer` or Kafka publish spikes point to async follow-up work.

6. **If the route family is `live` or `vod`, inspect origin**
   - Use the origin grouped queries in this notebook.
   - If origin p95 rises with upstream p95, origin is the likely bottleneck.
   - If origin stays low but edge stays high, look at cache behavior, lock contention, or edge overhead.

7. **Validate with raw edge access logs**
   - Logs tell you whether a specific slow request was mostly upstream time, edge overhead, or OpenResty internal work.
   - Use the `likely_request_bottleneck` field in the log analysis table as a starting heuristic, not as absolute truth.

8. **Use the Kafka mock only as a sink timing check**
   - It proves async publish timing and payload content.
   - It is not a real Kafka durability or broker-behavior test.

### Mental model

- `k6` tells you what the client observed.
- `openresty_edge_request_duration_seconds` tells you what the edge spent overall.
- `openresty_edge_upstream_response_seconds` tells you how much of that was upstream wait.
- `openresty_edge_lua_*`, metadata, Redis, and Kafka metrics tell you what happened inside the edge.
- `origin_request_latency_seconds` tells you what the origin itself spent.

If you keep that layered model in mind, you can usually localize a slowdown in a few minutes.